# Bước 5 - Chuẩn hóa kiểu dữ liệu và mã hóa

**Notebook:** `05_data_encoding.ipynb`  
**Input:** `../data/processed/step4_nooutliers.csv`  
**Output:** `../data/processed/step5_encoded.csv`

## Mục tiêu

Bước này chuẩn bị dữ liệu sau xử lý ngoại lai cho bước tạo RFM và các mô hình phân tích tiếp theo.

Các công việc chính:
1. Đọc dữ liệu đầu ra của Bước 4.
2. Kiểm tra kiểu dữ liệu và missing values còn lại.
3. Chuyển các cột ngày tháng sang kiểu `datetime64[ns]`.
4. Mã hóa các biến phân loại phù hợp, không mã hóa máy móc tất cả cột chữ.
5. Kiểm tra dữ liệu sau xử lý và xuất file cho Bước 6.

> **Phạm vi:** Notebook này không xử lý missing values, duplicates, outliers hay RFM. Những phần đó thuộc các bước khác trong quy trình.


## Phần 2 - Import thư viện

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)


## Phần 3 - Đọc dữ liệu

Đọc dữ liệu đã xử lý ngoại lai từ Bước 4. Sử dụng đường dẫn tương đối vì notebook nằm trong thư mục `notebooks/`.

In [2]:
input_path = '../data/processed/step4_nooutliers.csv'

df = pd.read_csv(input_path)
print(f"Đã đọc dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")


Đã đọc dữ liệu: 95075 dòng, 26 cột


In [3]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0


## Phần 4 - Kiểm tra dữ liệu đầu vào

Kiểm tra shape, kiểu dữ liệu và missing values trước khi chuyển đổi. Một số cột ngày giao hàng vẫn có missing vì đơn hàng chưa giao/hủy; bước này chỉ ghi nhận, không tự ý xóa.

In [4]:
print("Shape:", df.shape)
df.info()


Shape: (95075, 26)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95075 entries, 0 to 95074
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       95075 non-null  object 
 1   customer_id                    95075 non-null  object 
 2   order_status                   95075 non-null  object 
 3   order_purchase_timestamp       95075 non-null  object 
 4   order_approved_at              95062 non-null  object 
 5   order_delivered_carrier_date   94162 non-null  object 
 6   order_delivered_customer_date  93126 non-null  object 
 7   order_estimated_delivery_date  95075 non-null  object 
 8   order_item_id                  95075 non-null  int64  
 9   product_id                     95075 non-null  object 
 10  seller_id                      95075 non-null  object 
 11  shipping_limit_date            95075 non-null  object 
 12  price                          95075 non-null 

In [5]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
})
missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_percent', ascending=False)


,missing_count,missing_percent
order_delivered_customer_date,1949,2.049961
order_delivered_carrier_date,913,0.960295
order_approved_at,13,0.013673


## Phần 5 - Chuyển đổi kiểu dữ liệu thời gian

Các cột thời gian đang được đọc dưới dạng `object`. Chuyển sang `datetime64[ns]` để Bước 6 có thể tính Recency chính xác. Missing ở các mốc giao hàng được giữ nguyên vì có ý nghĩa nghiệp vụ.

In [6]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date'
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

for col in datetime_cols:
    print(f"{col}: {df[col].dtype}, missing/invalid sau chuyển đổi = {df[col].isna().sum()}")


order_purchase_timestamp: datetime64[ns], missing/invalid sau chuyển đổi = 0
order_approved_at: datetime64[ns], missing/invalid sau chuyển đổi = 13
order_delivered_carrier_date: datetime64[ns], missing/invalid sau chuyển đổi = 913
order_delivered_customer_date: datetime64[ns], missing/invalid sau chuyển đổi = 1949
order_estimated_delivery_date: datetime64[ns], missing/invalid sau chuyển đổi = 0
shipping_limit_date: datetime64[ns], missing/invalid sau chuyển đổi = 0


In [7]:
print("Số giá trị order_purchase_timestamp không hợp lệ:", df['order_purchase_timestamp'].isna().sum())
print("Khoảng thời gian mua hàng:")
print("  Từ :", df['order_purchase_timestamp'].min())
print("  Đến:", df['order_purchase_timestamp'].max())


Số giá trị order_purchase_timestamp không hợp lệ: 0
Khoảng thời gian mua hàng:
  Từ : 2016-09-04 21:15:19
  Đến: 2018-09-03 09:06:57


## Phần 6 - Khảo sát biến phân loại

Không mã hóa toàn bộ cột kiểu chữ vì nhiều cột là định danh (`order_id`, `customer_id`, `product_id`, `seller_id`) hoặc có quá nhiều nhóm (`customer_city`).

In [8]:
object_cols = df.select_dtypes(include='object').columns.tolist()
cardinality = pd.DataFrame({
    'column': object_cols,
    'unique_count': [df[col].nunique() for col in object_cols]
}).sort_values('unique_count', ascending=False)
cardinality


,column,unique_count
0,order_id,82936
1,customer_id,82936
5,customer_unique_id,80332
3,product_id,27569
6,customer_city,3533
4,seller_id,2789
8,product_category_name,74
7,customer_state,27
2,order_status,7


In [9]:
categorical_cols = ['customer_state', 'order_status', 'product_category_name']

for col in categorical_cols:
    print(f"\n{col} - số nhóm: {df[col].nunique()}")
    print(df[col].value_counts().head(10))



customer_state - số nhóm: 27
customer_state
SP    42708
RJ    12347
MG    11440
RS     5286
PR     4924
SC     3549
BA     2940
DF     2061
ES     1912
GO     1892
Name: count, dtype: int64

order_status - số nhóm: 7
order_status
delivered      93127
shipped          972
canceled         421
invoiced         287
processing       260
unavailable        5
approved           3
Name: count, dtype: int64

product_category_name - số nhóm: 74
product_category_name
cama_mesa_banho           10321
beleza_saude               8059
esporte_lazer              7601
moveis_decoracao           7143
informatica_acessorios     6775
utilidades_domesticas      6053
relogios_presentes         4560
telefonia                  4161
brinquedos                 3547
automotivo                 3473
Name: count, dtype: int64


## Phần 7 - Quyết định mã hóa

| Biến | Cách xử lý | Lý do |
|---|---|---|
| `customer_state` | One-Hot Encoding | 27 bang, số nhóm vừa phải và không có thứ tự tự nhiên |
| `order_status` | One-Hot Encoding | 7 trạng thái, số nhóm ít và phù hợp cho mô hình phân loại |
| `product_category_name` | Label Encoding | 74 danh mục; one-hot sẽ làm tăng nhiều cột, nên tạo mã số gọn hơn |
| `customer_city` | Không mã hóa | 3,533 thành phố, cardinality cao, dễ làm phình dữ liệu |
| Các cột ID | Không mã hóa | Là định danh, không mang ý nghĩa thứ bậc hay khoảng cách |

Các cột gốc vẫn được giữ lại để tiện giải thích và phục vụ các bước phân tích khác.

In [10]:
df_encoded = df.copy()

one_hot_cols = ['customer_state', 'order_status']
one_hot_encoded = pd.get_dummies(
    df_encoded[one_hot_cols],
    columns=one_hot_cols,
    prefix=one_hot_cols,
    drop_first=True,
    dtype=int
)

df_encoded = pd.concat([df_encoded, one_hot_encoded], axis=1)

label_encoder = LabelEncoder()
df_encoded['product_category_encoded'] = label_encoder.fit_transform(
    df_encoded['product_category_name'].astype(str)
)

category_mapping = pd.DataFrame({
    'product_category_name': label_encoder.classes_,
    'product_category_encoded': range(len(label_encoder.classes_))
})

print("Số cột trước encoding:", df.shape[1])
print("Số cột sau encoding  :", df_encoded.shape[1])
print("Số cột one-hot tạo thêm:", one_hot_encoded.shape[1])


Số cột trước encoding: 26
Số cột sau encoding  : 59
Số cột one-hot tạo thêm: 32


In [11]:
category_mapping.head(10)

,product_category_name,product_category_encoded
0,Unknown,0
1,agro_industria_e_comercio,1
2,alimentos,2
3,alimentos_bebidas,3
4,artes,4
5,artes_e_artesanato,5
6,artigos_de_festas,6
7,artigos_de_natal,7
8,audio,8
9,automotivo,9


## Phần 8 - Kiểm tra sau encoding

In [12]:
new_columns = [col for col in df_encoded.columns if col not in df.columns]
print("Các cột mới được tạo:")
print(new_columns[:20])
print("Tổng số cột mới:", len(new_columns))


Các cột mới được tạo:
['customer_state_AL', 'customer_state_AM', 'customer_state_AP', 'customer_state_BA', 'customer_state_CE', 'customer_state_DF', 'customer_state_ES', 'customer_state_GO', 'customer_state_MA', 'customer_state_MG', 'customer_state_MS', 'customer_state_MT', 'customer_state_PA', 'customer_state_PB', 'customer_state_PE', 'customer_state_PI', 'customer_state_PR', 'customer_state_RJ', 'customer_state_RN', 'customer_state_RO']
Tổng số cột mới: 33


In [13]:
required_for_rfm = ['customer_unique_id', 'order_id', 'order_purchase_timestamp', 'price']
missing_required = [col for col in required_for_rfm if col not in df_encoded.columns]

print("Thiếu cột bắt buộc cho RFM:", missing_required)
print("Datetime order_purchase_timestamp:", df_encoded['order_purchase_timestamp'].dtype)
print("Missing order_purchase_timestamp:", df_encoded['order_purchase_timestamp'].isna().sum())
print("Missing price:", df_encoded['price'].isna().sum())


Thiếu cột bắt buộc cho RFM: []
Datetime order_purchase_timestamp: datetime64[ns]
Missing order_purchase_timestamp: 0
Missing price: 0


In [14]:
print("Shape trước encoding:", df.shape)
print("Shape sau encoding  :", df_encoded.shape)
df_encoded.head()

Shape trước encoding: (95075, 26)
Shape sau encoding  : (95075, 59)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_state_AL,customer_state_AM,customer_state_AP,customer_state_BA,customer_state_CE,customer_state_DF,customer_state_ES,customer_state_GO,customer_state_MA,customer_state_MG,customer_state_MS,customer_state_MT,customer_state_PA,customer_state_PB,customer_state_PE,customer_state_PI,customer_state_PR,customer_state_RJ,customer_state_RN,customer_state_RO,customer_state_RR,customer_state_RS,customer_state_SC,customer_state_SE,customer_state_SP,customer_state_TO,order_status_canceled,order_status_delivered,order_status_invoiced,order_status_processing,order_status_shipped,order_status_unavailable,product_category_encoded
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,73
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,63
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,9
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,64
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,60


## Phần 9 - Xuất file

Lưu dữ liệu đã chuyển kiểu và mã hóa thành `step5_encoded.csv` cho Bước 6.

In [15]:
output_path = '../data/processed/step5_encoded.csv'

df_encoded.to_csv(output_path, index=False)

print(f"Đã lưu dữ liệu sau encoding vào: {output_path}")
print(f"Kích thước file output: {df_encoded.shape}")


Đã lưu dữ liệu sau encoding vào: ../data/processed/step5_encoded.csv
Kích thước file output: (95075, 59)


## Phần 10 - Nhận xét

- Đã chuyển các cột thời gian sang `datetime64[ns]`, đặc biệt là `order_purchase_timestamp` cho bước RFM.
- Đã one-hot `customer_state` và `order_status` vì số nhóm vừa phải.
- Đã label encode `product_category_name` để tránh tăng quá nhiều cột.
- Không mã hóa `customer_city` và các cột ID vì cardinality cao hoặc chỉ là định danh.
- Dataset `step5_encoded.csv` giữ đầy đủ các cột cần thiết cho Bước 6: `customer_unique_id`, `order_id`, `order_purchase_timestamp`, `price`.
